# Implied Volatility Surface

Option chain from `yfinance` -> implied volatility across strike and time, with Plotly.

Terms:
- strike - the price the option can be exercised at
- tenor (T) - time left to expiration, in years
- OTM - out-of-the-money: calls above spot, puts below spot
- implied volatility - the volatility implied by an option's market price

In [53]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
import plotly.graph_objects as go

symbol = "NVDA"               # ticker
strike_range = (200, 240)    # strike range
tenor_range = (0, 1)        # tenor range (years)

# Implied volatility from the option chain

Every option in the chain reports an implied volatility: the volatility implied by its market price. We read that number straight from the chain and plot it. No formula, no root-finding.

Black-Scholes uses one volatility for every strike and every date. Market quotes differ per strike and per date, so the result is a surface, not a single number.

In [54]:
tk = yf.Ticker(symbol)
S0 = tk.history(period="5d")["Close"].iloc[-1]     # last price
today = datetime.now().date()

rows = []
for exp in tk.options:
    d = pd.Timestamp(exp).date()
    days = (d - today).days
    T = days / 365.25
    if T < tenor_range[0] or T > tenor_range[1]:
        continue

    chain = tk.option_chain(str(d))
    for name, table in (("call", chain.calls), ("put", chain.puts)):
        for r in table.to_dict("records"):
            strike = r["strike"]
            iv = r["impliedVolatility"]     # implied volatility straight from the option chain
            # out-of-the-money only, sane volatility only
            if name == "call" and strike < S0:
                continue
            if name == "put" and strike >= S0:
                continue
            if strike < strike_range[0] or strike > strike_range[1]:
                continue
            if iv < 0.05 or iv > 0.60:
                continue
            rows.append([strike, iv, str(d), days])

df = pd.DataFrame(rows, columns=["strike", "iv", "expiration", "days"])
print("quotes kept:", len(df), "| expirations:", len(set(df["expiration"])))

quotes kept: 207 | expirations: 17


In [55]:
S0

np.float64(217.08999633789062)

# Time to expiration

We keep only expirations whose time to expiration $T = \text{days}/365.25$ falls inside `tenor_range`.

Each expiration quotes a different set of strikes (ragged chains). All expirations are placed on one common strike grid. Cells without quotes stay empty.

In [56]:
dates = sorted(df["expiration"].unique())                         # x-axis: expiration dates
K_grid = np.linspace(df["strike"].min(), df["strike"].max(), 60)  # y-axis: strike

# interpolate each expiration onto the strike grid
Z = np.full((60, len(dates)), np.nan)
for j, d in enumerate(dates):
    this = df[df["expiration"] == d].sort_values("strike")   # interp needs strikes in order
    Z[:, j] = np.interp(K_grid, this["strike"], this["iv"], left=np.nan, right=np.nan)

fig = go.Figure(go.Surface(
    x=[pd.Timestamp(d) for d in dates], y=K_grid, z=Z,
    colorscale="Viridis",
    hovertemplate="<b>%{x|%Y-%m-%d}</b><br>Strike: $%{y:.1f}<br>IV: %{z:.2%}<extra></extra>",
))
fig.update_layout(title=symbol, width=900, height=700)
fig.show()

# Skew and term structure

- strike direction = skew: puts usually trade at higher IV, so the low-strike side sits higher
- time direction = term structure: near-dated fear raises the front dates, far dates stay flat
- a kink in the curve = the market's view on where the move comes from

# Volatility smile

One expiration, one slice. OTM options are bid up, so the wings sit above the middle. A U-shape means the market prices fatter tails than Black-Scholes. A raised left side only is a smirk.

In [57]:
print("Available expirations (copy one into smile_date below):")
for d in sorted(df["expiration"].unique()):
    n = df[df["expiration"] == d]["days"].iloc[0]
    print("   ", d, "   ", n, "days")

Available expirations (copy one into smile_date below):
    2026-10-21     1 days
    2026-10-23     3 days
    2026-10-26     6 days
    2026-10-28     8 days
    2026-10-30     10 days
    2026-11-06     17 days
    2026-11-13     24 days
    2026-11-20     31 days
    2026-11-27     38 days
    2026-12-18     59 days
    2027-01-15     87 days
    2027-02-19     122 days
    2027-03-19     150 days
    2027-04-16     178 days
    2027-05-21     213 days
    2027-06-17     240 days
    2027-09-17     332 days


In [58]:
smile_date = "2026-10-21"      # paste any date from the list above

this = df[df["expiration"] == smile_date].sort_values(by="strike")

fig = go.Figure(go.Scatter(x=this["strike"], y=this["iv"], mode="markers+lines"))
fig.add_vline(x=S0, line_dash="dash")      # ATM strike line
fig.update_layout(title="{} smile - {}".format(symbol, smile_date),
                  xaxis_title="Strike", yaxis_title="Implied volatility",
                  width=900, height=500)
fig.show()

# Key points

- implied volatility comes from a market price
- the surface spans strikes and dates
- skew = what the market fears, term structure = when
- the smile = the market disagreeing with Black-Scholes